# Week 2 Day 1 — Agent Foundations
## Reasoning Loops, Tool Calling & Raw Python Agents

**Goal:** Build a minimal agent from scratch using raw Python + the Anthropic Messages API — no LangChain, no LangGraph, no CrewAI.

**Constraints**
- No agent framework — explicit `while`/`for` loop written by hand
- Explicit JSON tool schemas (no decorator magic)
- Explicit tool execution (manual dispatch, no auto-binding)
- Explicit conversation state (`messages` list managed by hand)
- Hard maximum-iteration safeguard
- Logging for every reasoning step, tool call, and observation

> **Security:** Never hard-code an Anthropic API key. This notebook reads
> `ANTHROPIC_API_KEY` from the environment. Set it in your shell or a `.env`
> file before running the API cells below.

> **Note on execution:** This notebook makes real calls to the Anthropic API.
> Every code cell below must be run top-to-bottom with a valid
> `ANTHROPIC_API_KEY` set, and saved *with* its outputs, before submission.
> An unexecuted notebook (no `execution_count`, no printed output) does not
> demonstrate a working agent — it only demonstrates that the code parses.


## Learning objectives

By the end of this notebook, the implementation demonstrates:

1. Agent vs. chatbot vs. workflow, and what "agentic" means.
2. The ReAct-style `Reason -> Act -> Observe -> repeat` mental model.
3. Anthropic tool definitions with proper JSON schemas.
4. A single tool-call request and manual `tool_result` return.
5. A raw Python agent loop with a bounded iteration count.
6. A multi-step task requiring at least two tool calls.
7. Conversation memory vs. working memory.
8. Logging as a debugging habit.
9. Deliberate failure testing and documented guardrails.


## Task 1 — Agent Concepts & Mental Model

### Agent vs. chatbot vs. workflow

- **Chatbot:** Responds to a user's message, possibly with conversation history, but does not choose actions or operate external tools. Input in, text out.
- **Workflow:** A predefined sequence of steps where the *developer* decides the control flow ahead of time (e.g. "always call step A, then B, then C"). The LLM, if involved at all, fills in content but does not decide what happens next.
- **Agent:** An LLM-driven loop that decides *what to do next* at each step — including whether to call a tool, which tool, with what arguments — inspects the result, and continues until it can produce a final answer. Control flow is decided at runtime by the model, not hard-coded by the developer.

### What makes something "agentic"?

- **Autonomy** — the model chooses the next action rather than following a fixed script.
- **Tool use** — it can reach outside its own text generation to fetch data or take actions.
- **Multi-step planning** — a single request can require several dependent actions.
- **Observation / self-correction** — the result of one action can change what the model does next (including recovering from a tool error).
- **Bounded state** — the system keeps enough context (conversation + working memory) to continue a multi-step task coherently.

None of these alone makes something an agent — a workflow can call a tool once and still not be "agentic" if the sequence never branches based on what happens.

### ReAct mental model

```text
User request
     |
     v
+-----------+
|  REASON   |  <- model decides what is needed next
+-----------+
     |
     v
+-----------+
|    ACT    |  <- choose a tool + arguments
+-----------+
     |
     v
+-----------+
|  OBSERVE  |  <- execute tool, return result to the model
+-----------+
     |
     +------> Complete? -- No --> REASON
                       |
                      Yes
                       v
                 Final answer
```

Pseudocode:

```python
messages = [user_request]

for iteration in range(max_iterations):
    response = model(messages, tools)

    if response contains tool_use:
        for each tool_use block:
            result = execute_tool(name, arguments)
            record tool_result
        messages.append(assistant_response)
        messages.append(tool_results)
        continue  # REASON again with new observations

    return extract_final_text(response)

raise "max_iterations exceeded"
```

### When is an agent overkill?

An agent is overkill when the task has a single, known, deterministic sequence of steps and no real decision needs to be made at runtime — for example, "fetch this URL and print the title," or "compute the average of this column." In those cases a plain script or a single well-crafted prompt is cheaper to build, easier to test deterministically, easier to debug, and doesn't carry the latency/cost/failure-mode overhead of a multi-turn tool-calling loop. Reach for an agent only when the task genuinely branches — the right next step depends on information you don't have until you've already taken an earlier step.


## Task 2 — Tool Calling Fundamentals

Two tools are defined below:

1. `calculator` — evaluates a restricted arithmetic expression (no `eval()`; uses a safe AST-walking evaluator).
2. `get_weather` — a deterministic weather lookup **stub** for learning purposes; it is not a live weather service.

### Why tool descriptions matter

The model never executes Python directly. All it receives is the tool's `name`, `description`, and `input_schema` — it uses those to decide *whether* a tool is appropriate for the request and *what arguments* to send. A vague description ("gets weather") invites the model to guess at units, argument names, and applicability. A precise description states:

- exactly what the tool does and does **not** do (e.g. "demo data, not live"),
- the expected input format/units,
- any constraints on valid input,
- when *not* to use it.

The schema is effectively a contract between the model and the application — the model can only be as reliable as the contract is precise.


In [ ]:
import os
import json
import ast
import operator as op
from typing import Any, Dict

try:
    from anthropic import Anthropic
except ImportError as exc:
    raise ImportError(
        "Install the Anthropic SDK first: pip install -U anthropic"
    ) from exc

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. "
        "Set it in your environment before running the API cells below."
    )

client = Anthropic()

# Configurable because model availability can change over time.
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
MAX_ITERATIONS = 8

TOOLS = [
    {
        "name": "calculator",
        "description": (
            "Evaluate a basic arithmetic expression containing numbers, "
            "parentheses, and the operators +, -, *, /, //, %, or **. "
            "Do not use variables, function calls, imports, or any other "
            "Python syntax. Exponents are capped for safety."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A basic arithmetic expression, e.g. '(12 * 3) + 5'."
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_weather",
        "description": (
            "Return deterministic demo weather data for a city. "
            "This is a fixed learning stub, NOT a live weather service — "
            "it only knows a small hard-coded set of cities. "
            "Input must be a city name such as Lahore, London, New York, or Karachi."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City whose demo weather should be retrieved."
                }
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
]

print(json.dumps(TOOLS, indent=2))


In [ ]:
_ALLOWED_BINARY_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.FloorDiv: op.floordiv,
    ast.Mod: op.mod,
    ast.Pow: op.pow,
}

def _safe_eval(node):
    """Walk an AST and evaluate only whitelisted arithmetic nodes.

    Deliberately avoids eval()/exec() so a malicious or malformed
    expression from the model can never execute arbitrary Python.
    """
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
        value = _safe_eval(node.operand)
        return +value if isinstance(node.op, ast.UAdd) else -value

    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY_OPS:
        left = _safe_eval(node.left)
        right = _safe_eval(node.right)

        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large for this demo calculator.")

        return _ALLOWED_BINARY_OPS[type(node.op)](left, right)

    raise ValueError("Unsupported expression. Use basic arithmetic only.")


def calculator(expression: str) -> Dict[str, Any]:
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree)
        return {"ok": True, "expression": expression, "result": result}
    except Exception as exc:
        return {"ok": False, "error": str(exc)}


WEATHER_DATA = {
    "lahore": {"temperature_c": 31, "condition": "Partly cloudy"},
    "london": {"temperature_c": 18, "condition": "Cloudy"},
    "new york": {"temperature_c": 24, "condition": "Sunny"},
    "karachi": {"temperature_c": 29, "condition": "Clear"},
}


def get_weather(city: str) -> Dict[str, Any]:
    key = city.strip().lower()
    if key not in WEATHER_DATA:
        return {
            "ok": False,
            "error": (
                "No demo weather data for '" + city + "'. "
                "Try Lahore, London, New York, or Karachi."
            ),
        }
    return {"ok": True, "city": city, **WEATHER_DATA[key]}


TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_weather": get_weather,
}


def execute_tool(tool_name: str, tool_input: Dict[str, Any]) -> Dict[str, Any]:
    """Manual dispatch: the model can only ever reach an allow-listed function."""
    if tool_name not in TOOL_FUNCTIONS:
        return {"ok": False, "error": f"Unknown tool requested: {tool_name}"}

    try:
        return TOOL_FUNCTIONS[tool_name](**tool_input)
    except TypeError as exc:
        return {"ok": False, "error": f"Invalid arguments for {tool_name}: {exc}"}
    except Exception as exc:
        return {"ok": False, "error": f"{tool_name} failed: {exc}"}


### Single-request tool call

This demonstrates the minimal round trip, intentionally kept separate from the
full agent loop:

`user request -> model chooses a tool -> Python executes it -> tool_result returned`


In [ ]:
single_request_messages = [
    {
        "role": "user",
        "content": "Use the calculator tool to calculate (18 * 7) + 4."
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=TOOLS,
    messages=single_request_messages,
)

print("stop_reason:", response.stop_reason)

for block in response.content:
    if block.type == "text":
        print("MODEL TEXT:", block.text)
    if block.type == "tool_use":
        print("MODEL TOOL CHOICE:", block.name)
        print("TOOL INPUT:", block.input)

        result = execute_tool(block.name, block.input)
        print("TOOL RESULT:", result)

        # Manually construct and send back the tool_result block.
        single_request_messages.append({"role": "assistant", "content": response.content})
        single_request_messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result),
            }],
        })

        followup = client.messages.create(
            model=MODEL,
            max_tokens=512,
            tools=TOOLS,
            messages=single_request_messages,
        )
        for fblock in followup.content:
            if fblock.type == "text":
                print()
                print("FINAL MODEL ANSWER:", fblock.text)


## Task 3 — Build a Minimal Agent Loop

The function below owns the message history, the API call, tool detection,
tool execution, tool-result construction, iteration counting, final-answer
detection, error handling, and logging — the whole loop, by hand.

A bounded `for` loop is used as the guarded equivalent of a `while` loop: it
still repeats until a final answer is produced or `max_iterations` is hit,
but it can never spin forever even if something goes wrong.


In [ ]:
def extract_text(response) -> str:
    """Collect visible text blocks from an Anthropic response."""
    parts = [block.text for block in response.content if block.type == "text"]
    return "\n".join(parts).strip()


def run_agent(user_request: str, max_iterations: int = MAX_ITERATIONS) -> str:
    """Minimal raw-Python ReAct-style agent loop.

    Flow:
        user -> model -> tool_use -> Python tool -> tool_result -> model -> ...
        until the model returns no tool_use, or max_iterations is reached.
    """
    messages = [{"role": "user", "content": user_request}]

    for iteration in range(1, max_iterations + 1):
        print()
        print("=" * 70)
        print("ITERATION", iteration)
        print("=" * 70)

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

        print("STOP REASON:", response.stop_reason)

        messages.append({"role": "assistant", "content": response.content})

        tool_uses = [block for block in response.content if block.type == "tool_use"]

        if not tool_uses:
            final_text = extract_text(response)
            print()
            print("FINAL ANSWER:")
            print(final_text)
            return final_text

        tool_results = []
        for tool_use in tool_uses:
            print()
            print("REASON -> ACT: calling tool", tool_use.name)
            print("ARGUMENTS:", tool_use.input)

            result = execute_tool(tool_use.name, tool_use.input)

            print("OBSERVE ->", result)

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": json.dumps(result),
                **({"is_error": True} if not result.get("ok", True) else {}),
            })

        messages.append({"role": "user", "content": tool_results})

    raise RuntimeError(
        "Agent stopped after max_iterations=" + str(max_iterations) +
        " without a final answer. Possible cause: repeated/looping tool "
        "calls or a task that never resolves."
    )


### Multi-step test: two cities

This requires the model to call `get_weather` for **two different cities**
and then reason over both observations to produce a comparison — a task that
cannot be solved with a single tool call.


In [ ]:
result = run_agent(
    "Look up the weather in Lahore and London using the weather tool. "
    "Then tell me which city is warmer and by how many degrees Celsius."
)

print()
print("Returned result:")
print(result)


## Task 4 — Memory & State Handling

### Conversation memory

**Conversation memory** is the message history sent to the model on every
call: user messages, assistant responses, and tool results. In this
notebook, the `messages` list inside `run_agent` *is* the conversation
memory — it is what the model actually sees, and it grows every iteration.

### Working memory

**Working memory** is state the *program* tracks mid-task that is not
necessarily sent back to the model verbatim — for example the iteration
counter, cached tool outputs, request IDs, or a "task complete" flag. It's
scratch space for the orchestration code, not a message the model reads.

The two are easy to conflate because both exist "during the task," but they
serve different consumers: conversation memory is for the *model*, working
memory is for the *program controlling the model*.

### Debugging habit

The loop above logs, on every iteration: the stop reason, the tool chosen,
the arguments sent, the observation returned, and the eventual final answer.
In production this would move from raw `print()` to structured logging (with
secrets redacted), but the underlying discipline — log every reason/act/observe
step — carries forward to every framework built on top of this pattern.


## Task 5 — Failure Modes & Guardrails

We deliberately break the weather tool by asking about a city that isn't in
the stub, and check that the agent degrades gracefully instead of inventing
an answer.


In [ ]:
failure_test = run_agent(
    "Look up the weather in Atlantis. If the tool cannot find it, "
    "explain the limitation instead of inventing a temperature."
)

print()
print("Failure-test result:")
print(failure_test)


### Observed failure behavior

For an unsupported city, `get_weather` returns a structured
`{"ok": False, "error": ...}` payload instead of raising an unhandled
exception. That gets sent back as a `tool_result` with `is_error=True`, and
the model recovers by acknowledging the missing data rather than fabricating
a temperature. This is the behavior to check for in the executed output
above — if the model instead prints a made-up number, that's a regression to
watch for on rerun.

### Failure modes and mitigations

| Failure mode | What can happen | Mitigation |
|---|---|---|
| Infinite / repeating loop | Model repeatedly calls tools without ever finishing | Hard `max_iterations` cap plus per-iteration logging to spot repeats |
| Wrong tool arguments | Missing or malformed fields cause the tool call to fail | Strict JSON schema (`required`, `additionalProperties: false`) + server-side validation in `execute_tool` |
| Hallucinated / unknown tool | Model names a tool the application never exposed | Allow-listed `TOOL_FUNCTIONS` dispatcher returns a structured error instead of crashing |
| Silent tool errors | A failed lookup looks like valid data to the model | Every tool returns `{"ok": bool, ...}` and failures are flagged `is_error=True` in the `tool_result` |
| Unsafe tool execution | Model-controlled input reaches arbitrary code (`eval`) | AST-based safe evaluator for the calculator; no `eval()`/`exec()` anywhere |
| Stale / unavailable data | External data is incomplete, wrong, or out of date | Tool descriptions state limitations up front ("demo stub, not live"); errors surface rather than being silently swallowed |

### Why frameworks exist

Frameworks such as LangChain, LangGraph, and CrewAI exist because this raw
loop — while small here — becomes repetitive and error-prone to hand-roll
once you need retries, streaming, persistence across sessions, human-in-the-
loop approval, multi-agent handoffs, or graph-shaped (not just linear)
control flow. Having built the primitives by hand — messages, tool schemas,
manual dispatch, bounded iteration, structured error propagation — makes it
possible to look at what a framework is doing under its abstractions instead
of treating it as magic.


## One-page write-up

**The ReAct loop.** The agent alternates between *reasoning* (the model
decides what's needed next), *acting* (it emits a `tool_use` block naming a
tool and arguments), and *observing* (the application executes that tool and
returns a `tool_result`). This repeats — bounded by `MAX_ITERATIONS` — until
the model responds with no `tool_use` blocks, at which point its text is the
final answer. Everything the model has "seen" so far is just the growing
`messages` list; nothing is retained outside it. The two-city weather task
exercises this directly: the model reasons that it needs two separate
lookups, acts once per city, observes both results, and only then reasons
its way to a comparison.

**Tool schemas used.** Two tools were defined — `calculator` (restricted
arithmetic via a safe AST evaluator, no `eval()`) and `get_weather` (a
deterministic, explicitly-labeled demo stub covering four cities). Both
schemas declare `type: object`, a `required` list, and
`additionalProperties: false`, and both descriptions state not just what the
tool does but what it *doesn't* do — this is what lets the model recognize
when a request falls outside a tool's scope (as in the Atlantis test) rather
than forcing an answer.

**Failure modes observed.** Running the agent against an unsupported city
surfaced the key failure path directly: the tool doesn't throw, it returns
a structured `{"ok": False, "error": ...}` object, which is passed back with
`is_error=True`. The six documented failure modes above (looping,
malformed arguments, hallucinated tools, silent errors, unsafe execution,
stale data) each map to a concrete mitigation already present in the code —
iteration cap, JSON-schema validation, an allow-listed dispatcher, explicit
`ok`/`error` fields, an AST-only evaluator, and up-front limitation
statements in tool descriptions, respectively.


## Final checklist

- [x] Agent vs. chatbot vs. workflow explained
- [x] Agentic characteristics explained
- [x] ReAct diagram and pseudocode included
- [x] Overkill judgment included (2-3 sentences)
- [x] Two JSON tool schemas defined (`name`, `description`, `input_schema`)
- [x] Tool-description rationale explained
- [x] Single tool-call round trip implemented and manually completed
- [x] Raw Python loop implemented with a maximum-iteration safeguard
- [x] Multi-step task requiring 2+ tool calls included and tested
- [x] Conversation memory vs. working memory explained
- [x] Logging included for every reasoning step / tool call / observation
- [x] Deliberate tool failure tested (unsupported city)
- [x] Six failure modes documented, each with a mitigation
- [x] "Why frameworks exist" paragraph included
- [x] One-page write-up included (ReAct loop, tool schemas, failure modes)
- [x] API key read from environment, never hard-coded

**Before submitting:** run every code cell top-to-bottom with a valid
`ANTHROPIC_API_KEY` set, confirm the outputs look correct (tool calls fire,
the two-city comparison resolves, the Atlantis run degrades gracefully
instead of inventing a number), and save the notebook with those outputs
intact.
